
# Exp.04A — Recover and Train the PGR²M Base Transformer

Exp.04R found **13 tokenizer checkpoints but no Base Transformer and no Refine Transformer**.
Therefore the conditioning intervention cannot yet be run with a matched generation stack.

This notebook rebuilds the first missing component:

> **Text + keywords → pose codes (Base Transformer)**

The tokenizer is frozen to the matched default configuration already used in the revision:

- HumanML3D
- residual dropout \(\tau=0.1\)
- entropy weight \(0.01\)
- seed 123
- best-validation-FID checkpoint from Exp.02

The Base Transformer follows the released README configuration:

- 9 Transformer layers
- 16 attention heads
- block size 62
- batch size 64
- 300k iterations
- LR \(1\times10^{-4}\)
- LR milestone 150k
- `pkeep=0.5`
- HumanML3D + GPT-4 keyword embeddings

Because `dataset_TM_train.py` is absent from the public PGR²M repository, this notebook creates
a **contract-compatible reconstruction based on the public CoMo loader**. The reconstruction is
clearly marked and does not claim to be the authors' unreleased private file.

The notebook performs:

1. environment/data/tokenizer hard gates;
2. loader reconstruction only if missing;
3. loader contract smoke;
4. one-iteration production-architecture smoke;
5. detached 300k Base Transformer launch;
6. read-only progress monitoring;
7. completion/checkpoint audit.

After Exp.04A completes, Exp.04B will reconstruct the residual-token loaders and train the
Refine Transformer against this exact Base Transformer + tokenizer pair.


In [ ]:

from pathlib import Path
import os, sys, json, re, time, hashlib, subprocess, shlex, textwrap
import numpy as np
import pandas as pd

PGR = Path(".")
PY = Path("venv/bin/python")
DATA = PGR / "dataset" / "HumanML3D"
PRE = PGR / "revision_preflight"
LOGDIR = PGR / "revision_logs" / "exp04a_base"
RESULTS = PGR / "revision_results" / "exp04a_base"
OUT_ROOT = PGR / "exp_result_revision_base_transformer"

EXP02_STATE = PGR / "revision_logs" / "exp02_tau" / "exp02_tau_queue_state.json"
EXP02_ROOT = PGR / "exp_result_revision_tau_sensitivity"
TRAIN_SRC = PGR / "train_base_trans.py"

LOADER = PGR / "dataset" / "dataset_TM_train.py"
EVAL_LOADER = PGR / "dataset" / "dataset_TM_eval.py"
WRAPPER = PRE / "train_base_trans_exp04a_compat.py"

LAUNCH = LOGDIR / "exp04a_base_launch.json"
LOG = LOGDIR / "exp04a_base_train.log"
PIDFILE = LOGDIR / "exp04a_base.pid"
FREEZE = RESULTS / "exp04a_base_config_freeze.json"
AUDIT = RESULTS / "exp04a_base_completion_audit.json"

for d in [PRE, LOGDIR, RESULTS, OUT_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1024*1024), b""):
            h.update(b)
    return h.hexdigest()

print("=== EXP.04A ENVIRONMENT ===")
print("Python :", sys.executable)
print("Expected:", PY)
print("PGR    :", PGR)

assert PGR.exists(), PGR
assert PY.exists(), PY
assert TRAIN_SRC.exists(), TRAIN_SRC
assert EVAL_LOADER.exists(), (
    f"Reconstructed dataset_TM_eval.py is missing: {EVAL_LOADER}"
)
assert DATA.exists(), DATA

for required in [
    DATA / "new_joint_vecs",
    DATA / "texts",
    DATA / "codes",
    DATA / "keyword_embeddings",
    DATA / "train.txt",
    DATA / "val.txt",
    DATA / "Mean.npy",
    DATA / "Std.npy",
]:
    assert required.exists(), required

if Path(sys.executable).resolve() != PY.resolve():
    print("[WARN] Please select the Python 3 kernel.")

print("[OK] Environment/data hard gate passed.")


In [ ]:

# 2. Resolve and freeze the matched default tokenizer from Exp.02 (tau=0.1).

assert EXP02_STATE.exists(), EXP02_STATE
state = json.loads(EXP02_STATE.read_text())
rec = state.get("runs", {}).get("0.1", {})
assert rec.get("status") == "complete", rec

base = EXP02_ROOT / "exp_pg_tokenizer_tau_0p1_seed123"
ckpts = sorted(base.rglob("net_best_fid.pth"), key=lambda p: p.stat().st_mtime)
assert ckpts, f"No Exp.02 tau=0.1 best-FID checkpoint under {base}"

TOKENIZER_CKPT = ckpts[-1]
TOKENIZER = TOKENIZER_CKPT.parent
TOKENIZER_ARGS = TOKENIZER / "arguments.yaml"

cfg = json.loads(TOKENIZER_ARGS.read_text())
assert abs(float(cfg["pdrop_res"]) - 0.1) < 1e-12
assert abs(float(cfg["params_soft_ent_loss"]) - 0.01) < 1e-12
assert int(cfg["rvq_num_quantizers"]) == 6
assert int(cfg["rvq_nb_code"]) == 512
assert int(cfg["seed"]) == 123

print("=== FROZEN TOKENIZER ===")
print("Run dir   :", TOKENIZER)
print("Checkpoint:", TOKENIZER_CKPT)
print("SHA256    :", sha256(TOKENIZER_CKPT))
print("tau       :", cfg["pdrop_res"])
print("lambda_ent:", cfg["params_soft_ent_loss"])
print("[OK] Matched default tokenizer frozen.")


In [ ]:

# 3. Reconstruct dataset_TM_train.py only if it is missing.
#
# Provenance:
# - public CoMo Text2MotionDataset contract
# - adapted to the recovered PGR2M/HumanML3D layout
# - uses finalized HumanML3D Mean.npy / Std.npy
# - accepts PGR2M's `use_keywords` argument
#
# This is an operational reconstruction, NOT a claim of recovering the private original.

loader_text = r"""
import torch
from torch.utils import data
import numpy as np
from os.path import join as pjoin
import random
import codecs as cs
from torch.utils.data._utils.collate import default_collate

def collate_fn(batch):
    batch.sort(key=lambda x: x[2], reverse=True)
    return default_collate(batch)

class Text2MotionDataset(data.Dataset):
    def __init__(self, dataset_name, unit_length=4, codebook_size=392, use_keywords=False):
        self.pointer = 0
        self.dataset_name = dataset_name
        self.codebook_size = codebook_size
        self.unit_length = unit_length
        self.use_keywords = use_keywords

        self.mot_end_idx = codebook_size
        self.mot_pad_idx = codebook_size + 1

        if dataset_name == "t2m":
            self.data_root = "./dataset/HumanML3D"
            self.joints_num = 22
            self.fps = 20
            self.dim_pose = 263
        elif dataset_name == "kit":
            self.data_root = "./dataset/KIT-ML"
            self.joints_num = 21
            self.fps = 12.5
            self.dim_pose = 251
        else:
            raise ValueError(dataset_name)

        self.motion_dir = pjoin(self.data_root, "new_joint_vecs")
        self.text_dir = pjoin(self.data_root, "texts")
        self.code_dir = pjoin(self.data_root, "codes")
        self.keyword_dir = pjoin(self.data_root, "keyword_embeddings")

        # Pose-code sequence after x4 temporal downsampling plus END.
        self.max_motion_length = 26 if unit_length == 8 else 51

        self.mean = np.load(pjoin(self.data_root, "Mean.npy"))
        self.std = np.load(pjoin(self.data_root, "Std.npy"))

        split_file = pjoin(self.data_root, "train.txt")
        with cs.open(split_file, "r") as f:
            id_list = [line.strip() for line in f if line.strip()]

        data_dict = {}
        new_name_list = []

        min_motion_len = 40 if dataset_name == "t2m" else 24

        for name in id_list:
            try:
                motion_full = np.load(pjoin(self.motion_dir, name + ".npy"))
                pose_full = np.load(pjoin(self.code_dir, name + ".npy")).astype(np.float32)
                keywords = np.load(pjoin(self.keyword_dir, name + ".npy")).astype(np.float32)

                if len(motion_full) < min_motion_len or len(motion_full) >= 200:
                    continue

                text_data = []
                has_full = False

                with cs.open(pjoin(self.text_dir, name + ".txt"), "r") as f:
                    for line in f:
                        parts = line.strip().split("#")
                        if len(parts) < 4:
                            continue

                        caption = parts[0]
                        tokens = parts[1].split(" ")
                        f_tag = float(parts[2])
                        to_tag = float(parts[3])
                        f_tag = 0.0 if np.isnan(f_tag) else f_tag
                        to_tag = 0.0 if np.isnan(to_tag) else to_tag

                        td = {"caption": caption, "tokens": tokens}

                        if f_tag == 0.0 and to_tag == 0.0:
                            has_full = True
                            text_data.append(td)
                        else:
                            a = int(f_tag * self.fps)
                            b = int(to_tag * self.fps)
                            n_motion = motion_full[a:b]
                            n_pose = pose_full[a:b]

                            if len(n_motion) < min_motion_len or len(n_motion) >= 200:
                                continue

                            new_name = f"{name}_{f_tag:.3f}_{to_tag:.3f}"
                            data_dict[new_name] = {
                                "motion": n_motion,
                                "pose": n_pose,
                                "text": [td],
                                "keywords": keywords,
                            }
                            new_name_list.append(new_name)

                if has_full and text_data:
                    data_dict[name] = {
                        "motion": motion_full,
                        "pose": pose_full,
                        "text": text_data,
                        "keywords": keywords,
                    }
                    new_name_list.append(name)

            except Exception:
                # Match the released loaders' permissive behavior on malformed samples.
                continue

        if not new_name_list:
            raise RuntimeError("dataset_TM_train reconstruction found zero training examples.")

        self.data_dict = data_dict
        self.name_list = new_name_list

    def __len__(self):
        return len(self.name_list)

    def __getitem__(self, item):
        d = self.data_dict[self.name_list[item]]
        motion = d["motion"].copy()
        pose = d["pose"].copy()
        text_data = random.choice(d["text"])
        caption = text_data["caption"]
        keywords = d["keywords"]

        # Align to temporal downsampling factor.
        m_length = (len(motion) // self.unit_length) * self.unit_length
        if m_length < self.unit_length:
            return self.__getitem__((item + 1) % len(self))

        start = random.randint(0, len(motion) - m_length)
        motion = motion[start:start + m_length]
        pose = pose[start:start + m_length][::self.unit_length]

        # CoMo-style occasional one-token crop.
        coin = np.random.choice([False, False, True])
        if coin and len(pose) > 1:
            if np.random.choice([True, False]):
                pose = pose[:-1]
                motion = motion[:-self.unit_length]
            else:
                pose = pose[1:]
                motion = motion[self.unit_length:]

        # Keyword sampling: one embedding from each group of 3.
        if self.use_keywords:
            keyword_indices = np.arange(0, 33, 3) + np.random.randint(0, 3, size=(11,))
            keyword_data = torch.from_numpy(keywords[keyword_indices].copy())

            # Released CoMo behavior randomly masks a subset of keywords.
            proba = float(np.random.rand())
            mask = torch.bernoulli(proba * torch.ones((11,))).bool()
            keyword_data[mask] = 0
        else:
            keyword_data = torch.zeros((11, keywords.shape[-1]), dtype=torch.float32)

        motion = (motion - self.mean) / self.std
        motion_len = len(motion)

        max_frames = 200
        if motion_len < max_frames:
            motion = np.concatenate(
                [motion, np.zeros((max_frames - motion_len, motion.shape[1]), dtype=motion.dtype)],
                axis=0,
            )

        token_len = pose.shape[0]

        # Base Transformer target: pose codes + END/PAD channels.
        end_token = np.zeros((1, self.codebook_size + 2), dtype=np.float32)
        end_token[:, self.mot_end_idx] = 1.0

        pose_ext = np.concatenate(
            [pose.astype(np.float32), np.zeros((token_len, 2), dtype=np.float32)],
            axis=1,
        )

        if token_len + 1 < self.max_motion_length:
            pad_n = self.max_motion_length - 1 - token_len
            pad = np.zeros((pad_n, self.codebook_size + 2), dtype=np.float32)
            pad[:, self.mot_pad_idx] = 1.0
            tokens = np.concatenate([pose_ext, end_token, pad], axis=0)
        else:
            # Defensive crop; HumanML3D <200 frames should normally fit.
            pose_ext = pose_ext[:self.max_motion_length - 1]
            token_len = len(pose_ext)
            tokens = np.concatenate([pose_ext, end_token], axis=0)

        return (
            caption,
            tokens.astype(np.float32),
            int(token_len),
            keyword_data.float(),
            motion.astype(np.float32),
            int(motion_len),
        )

def DATALoader(dataset_name, batch_size, codebook_size, unit_length=4,
               num_workers=8, use_keywords=False):
    return torch.utils.data.DataLoader(
        Text2MotionDataset(
            dataset_name,
            unit_length=unit_length,
            codebook_size=codebook_size,
            use_keywords=use_keywords,
        ),
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        collate_fn=collate_fn,
        drop_last=True,
    )

def cycle(iterable):
    while True:
        for x in iterable:
            yield x
"""

if LOADER.exists():
    print("[KEEP] Existing dataset_TM_train.py:", LOADER)
else:
    LOADER.write_text(loader_text)
    print("[WRITE] Reconstructed dataset_TM_train.py:", LOADER)

compile(LOADER.read_text(), str(LOADER), "exec")
print("[OK] Loader syntax passed.")


In [ ]:

# 4. Loader contract smoke.

smoke = r'''
import os, sys
from pathlib import Path
os.chdir(".")
sys.path.insert(0, ".")

from dataset import dataset_TM_train

dl = dataset_TM_train.DATALoader(
    "t2m", 4, 392,
    unit_length=4,
    num_workers=0,
    use_keywords=True,
)
batch = next(iter(dl))
caption, tokens, token_len, keywords, motion, motion_len = batch

print("[RUN] captions", len(caption))
print("[RUN] tokens", tuple(tokens.shape), tokens.dtype)
print("[RUN] token_len", tuple(token_len.shape), int(token_len.min()), int(token_len.max()))
print("[RUN] keywords", tuple(keywords.shape), keywords.dtype)
print("[RUN] motion", tuple(motion.shape), motion.dtype)
print("[RUN] motion_len", tuple(motion_len.shape), int(motion_len.min()), int(motion_len.max()))

assert tokens.shape[1:] == (51, 394), tokens.shape
assert keywords.shape[1:] == (11, 512), keywords.shape
assert motion.shape[1:] == (200, 263), motion.shape
assert tokens.isfinite().all()
assert keywords.isfinite().all()
assert motion.isfinite().all()
print("[OK] dataset_TM_train contract passed.")
'''

p = subprocess.run(
    [str(PY), "-c", smoke],
    cwd=PGR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    timeout=1800,
)
print(p.stdout)
if p.returncode != 0:
    raise RuntimeError("dataset_TM_train contract smoke failed.")


In [ ]:
# 5. Create a compatibility wrapper for the released train_base_trans.py.
#
# Public PGR2M eval_trans.py imports `get_subsamples_loader` from two dataset modules,
# but the helper was not released. More importantly, evaluation_transformer(draw=True)
# uses it only for qualitative visualization and then calls the legacy 3D plotting code,
# which is incompatible with current Matplotlib (`Axes3D.lines` has no setter).
#
# For training/revision experiments, qualitative TensorBoard video rendering is not
# needed for model selection. We therefore:
#   1) provide fail-fast import shims only so eval_trans can import;
#   2) wrap evaluation_transformer() and force draw=False.
#
# This preserves the full quantitative validation loop (FID/R-Precision/MM-Dist/
# Diversity) and best-FID checkpoint selection while skipping ONLY visualization.

CLIP_ROOT = PRE / "third_party" / "openai_clip_dcba3cb2e282"
assert CLIP_ROOT.exists(), CLIP_ROOT

wrapper_text = r'''
import os, sys, runpy, types
from pathlib import Path
import numpy as np

PGR = Path(".")
CLIP_ROOT = PGR / "revision_preflight" / "third_party" / "openai_clip_dcba3cb2e282"

os.chdir(PGR)
sys.path.insert(0, str(PGR))
sys.path.insert(0, str(CLIP_ROOT))

# NumPy 2.x compatibility used by older upstream code.
if not hasattr(np, "float"):
    np.float = float
if not hasattr(np, "int"):
    np.int = int
if not hasattr(np, "bool"):
    np.bool = bool

import ftfy
import clip
print("[OK] CLIP:", clip.__file__)

def _unreleased_subsamples_loader(*args, **kwargs):
    raise NotImplementedError(
        "get_subsamples_loader was not released publicly. "
        "Exp.04A disables qualitative drawing, so this helper must not be called."
    )

# Import shims required only because utils/eval_trans.py imports these symbols eagerly.
import dataset.dataset_TM_eval as _tm_eval
if not hasattr(_tm_eval, "get_subsamples_loader"):
    _tm_eval.get_subsamples_loader = _unreleased_subsamples_loader
    print("[OK] Added dataset_TM_eval.get_subsamples_loader import shim")

try:
    import dataset.dataset_RTM_eval as _rtm_eval
except ModuleNotFoundError:
    import dataset as _dataset_pkg
    _rtm_eval = types.ModuleType("dataset.dataset_RTM_eval")
    _rtm_eval.get_subsamples_loader = _unreleased_subsamples_loader
    sys.modules["dataset.dataset_RTM_eval"] = _rtm_eval
    setattr(_dataset_pkg, "dataset_RTM_eval", _rtm_eval)
    print("[OK] Added dataset_RTM_eval import-compatible module")
else:
    if not hasattr(_rtm_eval, "get_subsamples_loader"):
        _rtm_eval.get_subsamples_loader = _unreleased_subsamples_loader
        print("[OK] Added dataset_RTM_eval.get_subsamples_loader import shim")

# Import after shims are installed.
import utils.eval_trans as _eval_trans

_original_evaluation_transformer = _eval_trans.evaluation_transformer

def _evaluation_transformer_no_draw(*args, **kwargs):
    # Quantitative evaluation and checkpoint selection are unchanged.
    # Skip only the qualitative 3D/TensorBoard rendering path.
    kwargs["draw"] = False
    return _original_evaluation_transformer(*args, **kwargs)

_eval_trans.evaluation_transformer = _evaluation_transformer_no_draw
print("[OK] Forced evaluation_transformer(draw=False): quantitative metrics retained; visualization skipped")

runpy.run_path(str(PGR / "train_base_trans.py"), run_name="__main__")
'''

WRAPPER.write_text(wrapper_text)
compile(WRAPPER.read_text(), str(WRAPPER), "exec")
print("[OK] Wrapper:", WRAPPER)


In [ ]:

# 6. Freeze the production Base Transformer configuration.

EXP_NAME = "exp_base_trans_revision_seed123"
COMMON = [
    "--exp-name", EXP_NAME,
    "--batch-size", "64",
    "--num-layers", "9",
    "--nb-code", "392",
    "--n-head-gpt", "16",
    "--block-size", "62",
    "--ff-rate", "4",
    "--embed-dim-gpt", "1024",
    "--out-dir", str(OUT_ROOT),
    "--total-iter", "300000",
    "--lr-scheduler", "150000",
    "--lr", "0.0001",
    "--dataname", "t2m",
    "--down-t", "2",
    "--depth", "3",
    "--eval-iter", "10000",
    "--pkeep", "0.5",
    "--num_workers", "16",
    "--dilation-growth-rate", "3",
    "--output-emb-width", "392",
    "--val-shuffle",
    "--log-cat-right-num",
    "--eval-masking",
    "--min-sampling-prob", "0.1",
    "--dec-checkpoint-folder", str(TOKENIZER),
    "--use-keywords",
    "--seed", "123",
]

freeze = {
    "experiment": "Exp04A Base Transformer recovery",
    "tokenizer_dir": str(TOKENIZER),
    "tokenizer_checkpoint_sha256": sha256(TOKENIZER_CKPT),
    "train_source": str(TRAIN_SRC),
    "train_source_sha256": sha256(TRAIN_SRC),
    "dataset_TM_train": str(LOADER),
    "dataset_TM_train_sha256": sha256(LOADER),
    "dataset_TM_train_provenance": "contract-compatible reconstruction based on public CoMo loader",
    "dataset_TM_eval": str(EVAL_LOADER),
    "dataset_TM_eval_sha256": sha256(EVAL_LOADER),
    "args": COMMON,
}
FREEZE.write_text(json.dumps(freeze, indent=2))
print("[OK] Freeze:", FREEZE)
print(" ".join(shlex.quote(x) for x in [str(PY), "-u", str(WRAPPER), *COMMON]))


In [ ]:

# 7. One-iteration production-architecture smoke.
#    Uses the same 9-layer/16-head model and batch size 64.
#    The released script performs an initial validation before the first update,
#    so this can take several minutes.

SMOKE_OUT = PRE / "exp04a_base_smoke_out"
SMOKE_LOG = LOGDIR / "exp04a_base_smoke.log"

smoke_args = []
it = iter(COMMON)
pairs = list(zip(it, it))  # not used; rebuild explicitly below

SMOKE = [
    "--exp-name", "exp04a_base_smoke",
    "--batch-size", "64",
    "--num-layers", "9",
    "--nb-code", "392",
    "--n-head-gpt", "16",
    "--block-size", "62",
    "--ff-rate", "4",
    "--embed-dim-gpt", "1024",
    "--out-dir", str(SMOKE_OUT),
    "--total-iter", "1",
    "--lr-scheduler", "150000",
    "--lr", "0.0001",
    "--dataname", "t2m",
    "--down-t", "2",
    "--depth", "3",
    "--eval-iter", "1",
    "--pkeep", "0.5",
    "--num_workers", "4",
    "--dilation-growth-rate", "3",
    "--output-emb-width", "392",
    "--val-shuffle",
    "--log-cat-right-num",
    "--eval-masking",
    "--min-sampling-prob", "0.1",
    "--dec-checkpoint-folder", str(TOKENIZER),
    "--use-keywords",
    "--seed", "123",
]

cmd = [str(PY), "-u", str(WRAPPER), *SMOKE]

print("=== EXP.04A BASE SMOKE ===")
print(" ".join(shlex.quote(x) for x in cmd))

t0 = time.time()
with open(SMOKE_LOG, "w") as f:
    p = subprocess.run(
        cmd,
        cwd=PGR,
        stdout=f,
        stderr=subprocess.STDOUT,
        timeout=14400,
    )
elapsed = (time.time() - t0) / 60

txt = SMOKE_LOG.read_text(errors="replace")
print("returncode:", p.returncode)
print(f"elapsed: {elapsed:.1f} min")
print("\n".join(txt.splitlines()[-120:]))

if p.returncode != 0:
    raise RuntimeError(f"Base Transformer smoke failed: {SMOKE_LOG}")

markers = ["loading decoder checkpoint", "Train. Iter"]
missing = [m for m in markers if m.lower() not in txt.lower()]
if missing:
    raise RuntimeError(f"Smoke returned 0 but markers missing: {missing}")

print("[OK] Exp.04A production-architecture smoke passed.")


In [ ]:

# 8. Launch the 300k Base Transformer training as a detached process.
#    Safe PID guard prevents accidental duplicate launches.

def pid_alive(pid):
    try:
        os.kill(pid, 0)
        return True
    except Exception:
        return False

old_pid = None
if PIDFILE.exists():
    try:
        old_pid = int(PIDFILE.read_text().strip())
    except Exception:
        pass

RUN_ROOT = OUT_ROOT / EXP_NAME

# If already complete, do not relaunch.
complete_ckpts = list(RUN_ROOT.rglob("net_best_fid.pth")) if RUN_ROOT.exists() else []
log_text = LOG.read_text(errors="replace") if LOG.exists() else ""
already_300k = "Train. Iter 300000" in log_text

if already_300k and complete_ckpts:
    print("=== EXP.04A ALREADY COMPLETE ===")
    print("Checkpoint candidates:", len(complete_ckpts))
elif old_pid and pid_alive(old_pid):
    print("=== EXP.04A ALREADY RUNNING ===")
    print("PID  :", old_pid)
    print("Alive:", True)
else:
    cmd = [str(PY), "-u", str(WRAPPER), *COMMON]
    handle = open(LOG, "a")
    p = subprocess.Popen(
        cmd,
        cwd=PGR,
        stdout=handle,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    PIDFILE.write_text(str(p.pid))
    LAUNCH.write_text(json.dumps({
        "pid": p.pid,
        "log": str(LOG),
        "output_root": str(OUT_ROOT),
        "exp_name": EXP_NAME,
        "command": cmd,
        "launched_at": time.time(),
    }, indent=2))
    time.sleep(1)

    print("=== EXP.04A BASE TRAIN LAUNCH ===")
    print("PID   :", p.pid)
    print("Alive :", pid_alive(p.pid))
    print("Log   :", LOG)
    print("Output:", RUN_ROOT)


In [ ]:

# 9. Read-only progress status — safe to run at any time.

def max_train_iter(log_path):
    if not log_path.exists():
        return None
    pat = re.compile(r"Train\. Iter\s+(\d+)")
    last = None
    for line in log_path.read_text(errors="replace").splitlines():
        m = pat.search(line)
        if m:
            last = max(last or 0, int(m.group(1)))
    return last

pid = None
if PIDFILE.exists():
    try:
        pid = int(PIDFILE.read_text().strip())
    except Exception:
        pass

it = max_train_iter(LOG)
pct = 100.0 * it / 300000 if it is not None else 0.0

print("=== EXP.04A BASE STATUS ===")
print("PID   :", pid)
print("Alive :", pid_alive(pid) if pid else False)
print("Iter  :", it)
print(f"Progress: {pct:.2f}%")
print("Log   :", LOG)

print("\n=== LOG TAIL ===")
if LOG.exists():
    print("\n".join(LOG.read_text(errors="replace").splitlines()[-60:]))


In [ ]:

# 10. Completion/checkpoint audit.
#     Run only after status shows 300000.

it = max_train_iter(LOG)
if it is None or it < 300000:
    raise RuntimeError(f"Base Transformer not complete yet: max Train. Iter = {it}")

RUN_ROOT = OUT_ROOT / EXP_NAME
ckpts = sorted(RUN_ROOT.rglob("net_best_fid.pth"), key=lambda p: p.stat().st_mtime)
if not ckpts:
    raise RuntimeError(f"No net_best_fid.pth under {RUN_ROOT}")

BEST = ckpts[-1]
RUN_DIR = BEST.parent
ARGS = RUN_DIR / "arguments.yaml"

import torch
ck = torch.load(BEST, map_location="cpu")
nb_iter = ck.get("nb_iter")

audit = {
    "final_train_iter": int(it),
    "run_dir": str(RUN_DIR),
    "best_fid_checkpoint": str(BEST),
    "checkpoint_nb_iter": int(nb_iter) if nb_iter is not None else None,
    "checkpoint_sha256": sha256(BEST),
    "arguments_sha256": sha256(ARGS),
    "tokenizer_dir": str(TOKENIZER),
    "tokenizer_checkpoint_sha256": sha256(TOKENIZER_CKPT),
}
AUDIT.write_text(json.dumps(audit, indent=2))

print("=== EXP.04A COMPLETION AUDIT ===")
for k,v in audit.items():
    print(f"{k}: {v}")
print("[OK] Exp.04A Base Transformer training complete.")
print("Next: Exp.04B Refine Transformer loader reconstruction + training.")
